> Processing of the results from training (including *clusterization*)

In [1]:
import pathlib

import yaml

import pandas as pd
import numpy as np

import jax.numpy as jnp

from torch.utils.data import DataLoader

import trapallada.data
import trapallada.analysis
import trapallada.results.analysis

So that `import`ed Python modules are automatically reloaded when modified

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# load_dir

The base directory

In [3]:
base_dir = trapallada.util.get_base_directory()
base_dir

PosixPath('/home/manu/Sync/git/trapallada')

In [4]:
results_dir = base_dir / 'results' / 'multichannel' / 'rates_large_infs' / 'fft_z_30'
assert results_dir.exists()

From the results directory,
- the `Dataset`,
- the headers of the corresponding files,
- the ML model,
- a `DataFrame` with results from training, 
- a `dict` with the parameters

In [5]:
ds, positions_df, model, training_df, parameters = trapallada.results.analysis.load_dir(results_dir)

trapallada.data - INFO - computing the FFT of each signal...
trapallada.data - INFO - dismissing the right half of each FFT (length is now 113)...


For t-SNE

In [6]:
n_projected_components = 2

A `DataLoader` for the `Dataset`

In [7]:
dl = DataLoader(ds, batch_size=128, shuffle=False)

# project_onto_latent_space

Projection onto the latent space...

In [8]:
z = trapallada.results.analysis.project_onto_latent_space(dl, model.encoder)
z.shape

(30856, 30)

In [9]:
z_mean, z_sd = trapallada.results.analysis.project_onto_latent_space(dl, model.encoder, return_sd=True)
z_mean.shape, z_sd.shape

((30856, 30), (30856, 30))

In [10]:
type(z_mean), type(z_sd)

(jaxlib.xla_extension.ArrayImpl, jaxlib.xla_extension.ArrayImpl)

In [11]:
assert jnp.allclose(z, z_mean).item()

# wasserstein

In [12]:
d = trapallada.results.analysis.wasserstein_between_gaussians(z_mean, z_sd)
d.shape

(30856, 30856)

One might think of use JAX...

In [13]:
d_first = trapallada.results.analysis.wasserstein_between_one_and_many_gaussians(z_mean[0], z_sd[0], z_mean, z_sd)
assert (d_first == d[0]).all().item()
d_first

Array([0.       , 6.625867 , 1.6307664, ..., 6.017589 , 7.3090863,
       6.052107 ], dtype=float32)

...but it yields a memory error (and *chunking* doesn't seem to yield a boost)

In [14]:
# jax.vmap(wasserstein_between_one_and_many_gaussians, in_axes=(0, 0, None, None))(z_mean, z_sd, z_mean, z_sd)

In [16]:
del d

In [15]:
clusters = trapallada.results.analysis.wasserstein_cluster(z_mean, z_sd, n_clusters=4)
clusters.shape, np.unique(clusters)

KeyboardInterrupt: 

In [ ]:
# analysis_file = pathlib.Path('/home/manu/Sync/git/trapallada/results/batchsize_fix/rates_large_infs/time_z_4') / 'vae.parquet'
# analysis_file.exists()
# an = pd.read_parquet(analysis_file)

# z = an.filter(regex=r'z_\d+').values
# z_sd = an.filter(regex=r'z_sd_\d+').values
# z.shape, z_sd.shape

# trapallada.results.analysis.wasserstein_between_gaussians(z, z_sd)

# tsne_project

...which is, in turn, projected, for visualization, using t-SNE

In [11]:
tsne = trapallada.results.analysis.tsne_project(z, n_projected_components=n_projected_components)

A `DataFrame` with metadata

In [12]:
res_df = ds.metadata_df.copy()

Names of columns to be added to the `DataFrame` for the latent space and t-SNE

In [13]:
z_cols = [f'z_{i}' for i in range(1, z.shape[-1]+1)]
tsne_cols = [f'tsne_{i}' for i in range(1, n_projected_components+1)]

# hdbscan_cluster

Clustering using [HDBSCAN](https://github.com/scikit-learn-contrib/hdbscan) with pretty much every default parameter. See [this notebook](https://nbviewer.org/github/scikit-learn-contrib/hdbscan/blob/master/notebooks/How%20HDBSCAN%20Works.ipynb) for an explanation.

In [14]:
res_df['cluster in z'] = trapallada.results.analysis.hdbscan_cluster(z, 10)
res_df['cluster in projected space'] = trapallada.results.analysis.hdbscan_cluster(tsne, 80)

/home/manu/Sync/git/trapallada/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/manu/Sync/git/trapallada/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/manu/Sync/git/trapallada/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/manu/Sync/git/trapallada/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


# augment_with_positions

Position information extracted from the *headers* of the FITs files is merged with the above `DataFrame`

In [15]:
res_df = trapallada.results.analysis.augment_with_positions(res_df, positions_df)

# augment_with_simbad

Some *alleged* labels associated with spatial coordinates from SIMBAD (XMATCH) database are appended

In [16]:
catalog_file = base_dir / 'experiments' / 'Cross_Simbad_r3.5.csv'
    
res_df = trapallada.results.analysis.augment_with_simbad(res_df, catalog_file)

/home/manu/Sync/git/trapallada/trapallada/results/analysis.py:114: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  catalog_df = pd.read_csv(catalog_file)


# augment_with_window_and_signal_ids

Two new columns are added
- one just combines the *tar* and *FIT* file, which univocally determines the signal,
- the other yields the number of window *within* every (*tar* file, *FIT* file) combination

In [17]:
res_df = trapallada.results.analysis.augment_with_window_and_signal_ids(res_df)